# 06 — Autograd & nn Building Blocks

**Dataset**: `sklearn.datasets.load_digits` — 1797 handwritten digits, 8×8 grayscale images, 10 classes.  
**Goal**: This notebook is **PyTorch-only**. Practise: `requires_grad`, `.backward()`, computing  
gradients by hand vs autograd, `nn.Linear`, cross-entropy loss, and a single training step.

### Math Refresher

Given $\hat{y} = X W^T + b$, the MSE loss is:  
$$L = \frac{1}{N} \sum_i (\hat{y}_i - y_i)^2$$

Gradient w.r.t. $W$:  
$$\frac{\partial L}{\partial W} = \frac{2}{N} (\hat{y} - y)^T X$$

In [ ]:
# ── Shared Setup ────────────────────────────────────────────────────────────
from sklearn.datasets import load_digits
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

digits = load_digits()

np_images = digits.images.astype(np.float32)   # (1797, 8, 8)
np_labels = digits.target.astype(np.int64)     # (1797,)

# Flatten for linear models
np_flat = np_images.reshape(len(np_images), -1)  # (1797, 64)

pt_flat   = torch.tensor(np_flat)                # (1797, 64)
pt_labels = torch.tensor(np_labels)              # (1797,)

print(f'X: {pt_flat.shape}, y: {pt_labels.shape}')

---
## P1 — Basic Autograd: d(x²)/dx

Compute the gradient of $f(x) = x^2$ at $x = 3$ using autograd.  
The analytical answer is $f'(x) = 2x = 6$.

#### Drill — `basic_grad`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
# DRILL: forward y=x^2, backprop, check grad
y = x**2; y.backward()
assert x.grad.item() == 4.0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def basic_grad():
    """
    Returns:
        grad_val : float — should be 6.0
    """
    # Step 1: create a scalar tensor with gradient tracking
    x = ...   # torch.tensor(3.0, requires_grad=True)

    # Step 2: forward pass
    y = ...   # x ** 2

    # Step 3: backward pass — computes dy/dx
    ...       # y.backward()

    # Step 4: read the gradient
    grad_val = ...   # x.grad.item()

    return grad_val

g = basic_grad()

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.isclose(g, 6.0, atol=1e-5), f'Expected 6.0, got {g}'
print('P1 assertions passed ✓   d(x²)/dx at x=3 =', g)

---
## P2 — Autograd vs Hand-Computed MSE Gradient

Use a tiny linear model $\hat{y} = X W$ (no bias) on the digit images.  
Verify that autograd matches the analytical gradient.

In [ ]:
# ── NumPy Reference: Analytical Gradient ──────────────────────────────────────
np_X = np_flat.astype(np.float64)                              # (1797, 64)
np_y = np_labels.astype(np.float64).reshape(-1, 1)             # (1797, 1)

rng = np.random.default_rng(42)
np_W = rng.standard_normal((64, 1)).astype(np.float64) * 0.01  # (64, 1)

# Forward
np_pred  = np_X @ np_W                   # (1797, 1)
np_diff  = np_pred - np_y                # (1797, 1)
np_loss  = (np_diff ** 2).mean()         # scalar

# Analytical gradient: dL/dW = (2/N) * X^T @ (pred - y)
N = len(np_X)
np_dW = (2 / N) * (np_X.T @ np_diff)    # (64, 1)

print(f'NumPy loss: {np_loss:.6f}')
print(f'NumPy dW norm: {np.linalg.norm(np_dW):.6f}')

#### Drill — `autograd_vs_manual`
Practice the core operation before using it in the problem above.

In [ ]:
W = torch.tensor([1.0], requires_grad=True)
# DRILL: compute MSE and call backward()
loss = (W * 2 - 4)**2
loss.backward()
assert W.grad != 0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def autograd_vs_manual(X_np, y_np, W_np):
    """
    Replicate the MSE gradient using autograd.

    Returns:
        pt_loss   : float — should match np_loss
        pt_grad   : (64, 1) tensor — should match np_dW
    """
    tX = torch.tensor(X_np)            # (1797, 64)
    ty = torch.tensor(y_np)            # (1797, 1)

    # Mark W for gradient tracking
    tW = ...   # torch.tensor(W_np, requires_grad=True)

    # Forward: same math as NumPy
    t_pred = ...   # tX @ tW

    # MSE loss
    t_loss = ...   # ((t_pred - ty) ** 2).mean()

    # Backward: compute gradients
    ...            # t_loss.backward()

    # Inspect gradient
    pt_grad = ...  # tW.grad
    pt_loss = ...  # t_loss.detach().item()

    return pt_loss, pt_grad

pt_loss, pt_dW = autograd_vs_manual(np_X, np_y, np_W)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.isclose(np_loss, pt_loss, atol=1e-6), f'Loss mismatch: np={np_loss}, pt={pt_loss}'
assert np.allclose(np_dW, pt_dW.numpy(), atol=1e-5), 'Gradient mismatch!'
print('P2 assertions passed ✓  — autograd matches analytical gradient')

---
## P3 — torch.no_grad() and .detach()

Demonstrate when to use `torch.no_grad()` (inference) and `.detach()` (stop gradient flow).

#### Drill — `no_grad_demo`
Practice the core operation before using it in the problem above.

In [ ]:
x = torch.ones(2, requires_grad=True)
# DRILL: computation without tracking gradients
with torch.no_grad():
    y = x * 2
assert not y.requires_grad

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def no_grad_demo():
    """
    Returns:
        inference_output : (1797, 1) tensor — computed without grad tracking
        detached_loss    : float — detached from computation graph
    """
    tW = torch.randn(64, 1, requires_grad=True)

    # During inference/validation: no graph is built → faster + less memory
    with torch.no_grad():
        inference_output = ...   # pt_flat @ tW   → (1797, 1), no grad tracked

    # .detach() creates a new tensor that shares data but has no grad_fn
    loss = ((pt_flat @ tW) ** 2).mean()
    detached_loss = ...          # loss.detach().item()

    return inference_output, detached_loss

inf_out, det_loss = no_grad_demo()

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert inf_out.shape == (1797, 1)
assert inf_out.requires_grad == False, 'no_grad should disable gradient tracking'
assert isinstance(det_loss, float)
print('P3 assertions passed ✓')

---
## P4 — nn.Linear: 64 pixels → 10 classes

Create an `nn.Linear` layer that maps 64-d flat images to 10 class logits.  
This is the core of a simple digit classifier.

#### Drill — `build_linear_layer`
Practice the core operation before using it in the problem above.

In [ ]:
import torch.nn as nn
# DRILL: create nn.Linear from 3 to 2
layer = nn.Linear(3, 2)
assert layer.weight.shape == (2, 3)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def build_linear_layer():
    """
    Returns:
        layer   : nn.Linear(64, 10)
        logits  : (1797, 10) output tensor
    """
    # Step 1: create the layer
    # nn.Linear(in_features, out_features, bias=True)
    # Internally stores: weight (10, 64) and bias (10,)
    layer = ...   # nn.Linear(64, 10)

    # Step 2: forward pass — layer(X) computes X @ W^T + b
    logits = ...  # layer(pt_flat)   → (1797, 10)

    return layer, logits

linear_layer, logits = build_linear_layer()

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert isinstance(linear_layer, nn.Linear)
assert linear_layer.weight.shape == (10, 64)
assert linear_layer.bias.shape   == (10,)
assert logits.shape == (1797, 10)
# Output should have grad_fn since layer params require_grad
assert logits.grad_fn is not None, 'logits should be part of computation graph'
print('P4 assertions passed ✓')
print(f'  weight: {linear_layer.weight.shape}, bias: {linear_layer.bias.shape}')
print(f'  logits: {logits.shape}')

---
## P5 — Cross-Entropy Loss & Backward Pass

Compute cross-entropy loss on the logits and run the backward pass.  
`F.cross_entropy` expects:
- logits: `(N, C)` float
- targets: `(N,)` long (class indices, not one-hot)

#### Drill — `cross_entropy_backward`
Practice the core operation before using it in the problem above.

In [ ]:
import torch.nn as nn
logits = torch.zeros(1, 10, requires_grad=True)
target = torch.tensor([3])
# DRILL: CrossEntropyLoss calculation
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, target)
assert loss > 0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def cross_entropy_backward(layer, X, y):
    """
    Compute cross-entropy loss, backward pass, and inspect gradients.

    Returns:
        loss_val    : float — the loss value
        weight_grad : (10, 64) tensor — gradient of loss wrt layer.weight
        bias_grad   : (10,) tensor — gradient of loss wrt layer.bias
    """
    # Step 1: forward pass
    logits = layer(X)                     # (N, 10)

    # Step 2: compute cross-entropy loss
    # F.cross_entropy combines log_softmax + nll_loss
    loss = ...   # F.cross_entropy(logits, y)

    # Step 3: zero existing gradients (critical!)
    # Without this, gradients would ACCUMULATE from previous backward calls
    ...          # layer.zero_grad()

    # Step 4: backward pass
    ...          # loss.backward()

    # Step 5: read gradients
    weight_grad = ...   # layer.weight.grad
    bias_grad   = ...   # layer.bias.grad
    loss_val    = ...   # loss.item()

    return loss_val, weight_grad, bias_grad

loss_val, w_grad, b_grad = cross_entropy_backward(linear_layer, pt_flat, pt_labels)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert isinstance(loss_val, float)
assert loss_val > 0, 'Cross-entropy loss should be positive'
assert w_grad.shape == (10, 64)
assert b_grad.shape == (10,)
# Gradients should not be all zero (that would mean backward didn't run)
assert w_grad.abs().sum() > 0, 'Weight gradient should not be all zeros'
assert b_grad.abs().sum() > 0, 'Bias gradient should not be all zeros'
print(f'P5 assertions passed ✓  — loss: {loss_val:.4f}')

---
## P6 — Full Training Step with Optimizer

Combine everything: `nn.Linear` → `cross_entropy` → `backward` → `optimizer.step()`.  
Run 200 iterations and plot the loss curve.

#### Drill — `train_loop`
Practice the core operation before using it in the problem above.

In [ ]:
w = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)
# DRILL: zero_grad, backward, step
optimizer.zero_grad()
loss = (w - 3)**2
loss.backward()
optimizer.step()
assert w.item() > 1.0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def train_loop(X, y, n_steps=200, lr=0.01):
    """
    Train a linear classifier for n_steps.

    Returns:
        model     : the trained nn.Linear(64, 10)
        losses    : list[float] of length n_steps
        final_acc : float — accuracy on training data after training
    """
    torch.manual_seed(42)

    # Step 1: create model and optimizer
    model = ...       # nn.Linear(64, 10)
    optimizer = ...   # torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    for step in range(n_steps):
        # Step 2: forward
        logits = ...  # model(X)

        # Step 3: loss
        loss = ...    # F.cross_entropy(logits, y)

        # Step 4: backward
        ...           # optimizer.zero_grad()
        ...           # loss.backward()

        # Step 5: update weights
        ...           # optimizer.step()

        losses.append(loss.item())

    # Accuracy
    with torch.no_grad():
        preds = model(X).argmax(dim=1)
        final_acc = (preds == y).float().mean().item()

    return model, losses, final_acc

trained_model, losses, acc = train_loop(pt_flat, pt_labels)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert len(losses) == 200
assert losses[-1] < losses[0], 'Loss should decrease during training'
assert acc > 0.5, f'Accuracy should be > 50% for a simple linear model, got {acc:.2%}'
print(f'P6 assertions passed ✓')
print(f'  Initial loss: {losses[0]:.4f}  →  Final loss: {losses[-1]:.4f}')
print(f'  Training accuracy: {acc:.2%}')

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses, color='#3b82f6', linewidth=1.5)
plt.xlabel('Step'); plt.ylabel('Cross-Entropy Loss')
plt.title(f'Training Loss Curve — Final Accuracy: {acc:.1%}')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()